In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog
import seaborn as sns
import numpy as np

In [6]:
# Merge several dfs : 
all_dfs = [pd.read_csv(f) for f in ['data/20261001/measurements/cor_hor_1/cor_hor_1.csv',
                                   'data/20261001/measurements/cor_hor_2/cor_hor_2.csv',
                                   'data/20261001/measurements/cor_sag_lat_1.1/cor_sag_lat_1.1.csv',
                                   'data/20261001/measurements/cor_sag_lat_1.2/cor_sag_lat_1.2.csv',
                                   'data/20261001/measurements/cor_sag_lat_2.1/cor_sag_lat_2.csv',
                                   'data/20261001/measurements/cor_sag_mid_1/cor_sag_mid.csv',
                                   'data/20261001/measurements/med_hor_1/med_hor.csv',
                                   'data/20261001/measurements/med_sag_1/med_sag.csv']]
df = pd.concat(all_dfs).reset_index(drop=True)

display(df)

,Experiment Name,Folder,Well,Filename,GroupID,Time Stamp,Model of fitting,Discard Curve,Young's Modulus (Pa),Eff. Young's Modulus (Pa),...,Minimum depth (nm),Maximum depth (nm),k (N/m),R (μm),Header k (N/m),Header R (μm),Calibration Factor,Sample Radius (μm),Depth (nm),Duration (s)
0,cor_hor_1,Indentations,NaN,cor_hor_1 Indentation_001.txt,NaN,01-10-2026 12:36:02,Hertzian Contact,No,209.49,279.32,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,14.52
1,cor_hor_1,Indentations,NaN,cor_hor_1 Indentation_006.txt,NaN,01-10-2026 12:46:33,Hertzian Contact,Yes,NaN,NaN,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,14.82
2,cor_hor_1,Indentations,NaN,cor_hor_1 Indentation_007.txt,NaN,01-10-2026 12:50:42,Hertzian Contact,No,1222.26,1629.68,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,18.67
3,cor_hor_1,Indentations,NaN,cor_hor_1 Indentation_008.txt,NaN,01-10-2026 12:51:49,Hertzian Contact,No,1614.71,2152.95,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,18.66
4,cor_hor_1,Indentations,NaN,cor_hor_1 Indentation_009.txt,NaN,01-10-2026 12:52:55,Hertzian Contact,No,1685.99,2247.99,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,18.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,med_sag_1,matrix_scan01,NaN,med_sag_2 S-1 X-1 Y-1 I-1.txt,NaN,01-10-2026 14:48:03,Hertzian Contact,No,574.83,766.45,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,17.99
80,med_sag_1,matrix_scan01,NaN,med_sag_2 S-1 X-1 Y-3 I-1.txt,NaN,01-10-2026 14:49:31,Hertzian Contact,No,732.59,976.79,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,21.76
81,med_sag_1,matrix_scan01,NaN,med_sag_2 S-1 X-2 Y-1 I-1.txt,NaN,01-10-2026 14:51:42,Hertzian Contact,No,478.20,637.60,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,22.88
82,med_sag_2,matrix_scan01,NaN,med_sag_2 S-1 X-1 Y-2 I-1.txt,NaN,01-10-2026 15:25:45,Hertzian Contact,No,142.63,190.18,...,0.0,4240.0,0.49,26.5,0.49,26.5,NaN,NaN,0.0,13.53


In [7]:
# Change values from Pa to kPa in the columns "Young's Modulus (Pa)" and "Eff. Young's Modulus (Pa)"
df["Young's Modulus (Pa)"] = df["Young's Modulus (Pa)"] / 1000
df["Eff. Young's Modulus (Pa)"] = df["Eff. Young's Modulus (Pa)"] / 1000

# Remove flagged rows 
df = df[df['Discard Curve'] == "No"]

In [8]:
# Show E means grouped by "Experiment Name"
aggregated_df = df.groupby("Experiment Name").agg({"Young's Modulus (Pa)": ["mean", "std"], "Eff. Young's Modulus (Pa)": ["mean", "std"]})
display(aggregated_df.round(3))  # show less decimals

Young's Modulus (Pa)        Eff. Young's Modulus (Pa)       
                                mean    std                      mean    std
Experiment Name                                                             
cor_hor_1                      1.183  0.680                     1.577  0.907
cor_hor_2                      0.186    NaN                     0.248    NaN
cor_sag_lat_1.1                1.055  0.285                     1.407  0.380
cor_sag_lat_1.2                3.397  0.487                     4.529  0.650
cor_sag_lat_2.1                1.777  0.939                     2.369  1.252
cor_sag_lat_2.2                2.157  0.331                     2.876  0.441
cor_sag_mid_1                  1.254  1.018                     1.672  1.357
med_hor_1                      0.333  0.117                     0.444  0.156
med_sag_1                      0.595  0.128                     0.794  0.171
med_sag_2                      0.109  0.047                     0.145  0.063

In [9]:
# Change "Experiment Name " values to same value if it's name_1 and name_2, etc.r
df["Experiment Name"] = df["Experiment Name"].str.replace(r'(_\d+(?:\.\d+)*)$', '', regex=True)   

# Show E means grouped by "Experiment Name"
aggregated_df = df.groupby("Experiment Name").agg({"Young's Modulus (Pa)": ["mean", "std"], "Eff. Young's Modulus (Pa)": ["mean", "std"]})
display(aggregated_df.round(3))  # show less decimals


Young's Modulus (Pa)        Eff. Young's Modulus (Pa)       
                                mean    std                      mean    std
Experiment Name                                                             
cor_hor                        0.984  0.739                     1.312  0.985
cor_sag_lat                    1.962  0.979                     2.616  1.306
cor_sag_mid                    1.254  1.018                     1.672  1.357
med_hor                        0.333  0.117                     0.444  0.156
med_sag                        0.401  0.282                     0.534  0.376